# Notebook 1 — Latency KPI Forecasting
**Target KPI:** `one_way_latency_ms`

| Model | Why |
|---|---|
| **LSTM** | Strong baseline for sequential KPI data |
| **GRU** | Lighter than LSTM, fewer parameters |
| **TCN** | Dilated convolutions capture latency spikes at multiple time scales |

In [2]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [3]:
# ─────────────────────────────────────────────
#  CONFIGURATION  —  edit these values freely
# ─────────────────────────────────────────────

DATA_PATH   = "../Data/Model_data.csv"
OUTPUT_DIR  = "../outputs/latency"

TARGET_KPI  = "one_way_latency_ms"
TOLERANCE   = 2.0       # ± 2 ms for accuracy metric

SEQ_LEN     = 12        # look-back window (12 × 5 min = 1 hour)
TRAIN_RATIO = 0.8

HIDDEN_SIZE = 64
NUM_LAYERS  = 2
EPOCHS      = 10
BATCH_SIZE  = 2048
LR          = 1e-3

RNG_SEED = 42
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

Device: cpu


In [5]:
print("Loading data...")
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head(3)

Loading data...
Rows: 210,528 | Columns: 20


,timestamp,slice_type,latitude,longitude,one_way_latency_ms,jitter_ms,rtt_ms,packet_delay_budget_ms,handover_interruption_time_ms,reliability_percent,packet_loss_percent,packet_loss_rate_percent,bler_percent,throughput_dl_mbps,throughput_ul_mbps,spectral_efficiency_bps_hz,handover_success_rate_percent,energy_efficiency_bits_per_joule,anomaly,anomaly_type
0,2024-01-01 00:00:00,URLLC,33.800386,-7.547638,2.5865,0.5029,5.3423,0.7614,5.2166,99.9995,0.0005,0.0005,0.4930,106.5463,103.6793,9.9301,99.5036,476587788.0,0,normal
1,2024-01-01 00:05:00,URLLC,33.802700,-7.553952,2.4543,0.4950,5.1841,0.7626,5.0939,99.9995,0.0005,0.0005,0.4954,102.3002,102.2863,9.9559,99.4860,485576369.0,0,normal
2,2024-01-01 00:10:00,URLLC,33.800517,-7.556512,2.4245,0.4927,5.1083,0.7753,5.1232,99.9995,0.0005,0.0005,0.4994,97.0391,98.8266,9.9911,99.4985,490452024.0,0,normal


In [6]:
# Latency KPIs + lag features for this domain
FEATURE_COLS = [
    "one_way_latency_ms",
    "jitter_ms",
    "rtt_ms",
    "packet_delay_budget_ms",
    "bler_percent",
    "packet_loss_percent",
    "throughput_dl_mbps",
]

df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
print(f"Clean rows: {len(df):,}")

Clean rows: 210,528


In [7]:
print("\nBuilding sequences...")

X_raw = df[FEATURE_COLS].values
y_raw = df[TARGET_KPI].shift(-1).values   # predict t+1
X_raw, y_raw = X_raw[:-1], y_raw[:-1]    # drop last row (no target)

X_all, y_all = [], []
for i in range(SEQ_LEN, len(X_raw)):
    X_all.append(X_raw[i - SEQ_LEN : i, :])
    y_all.append(y_raw[i])

X_seq = np.stack(X_all).astype(np.float32)
y_seq = np.array(y_all, dtype=np.float32).reshape(-1, 1)
print(f"Sequences: {len(X_seq):,} | X: {X_seq.shape} | y: {y_seq.shape}")


Building sequences...
Sequences: 210,515 | X: (210515, 12, 7) | y: (210515, 1)


In [8]:
train_n = int(len(X_seq) * TRAIN_RATIO)
X_train, X_test = X_seq[:train_n], X_seq[train_n:]
y_train, y_test = y_seq[:train_n], y_seq[train_n:]

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[2])).reshape(X_train.shape)
X_test_scaled  = scaler_X.transform(X_test.reshape(-1, X_test.shape[2])).reshape(X_test.shape)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32, device=DEVICE)
X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32, device=DEVICE)
y_test_t  = torch.tensor(y_test_scaled,  dtype=torch.float32, device=DEVICE)

print(f"Train: {len(X_train_t):,} | Test: {len(X_test_t):,} | Features: {X_train_t.shape[2]} | Device: {DEVICE}")

Train: 168,412 | Test: 42,103 | Features: 7 | Device: cpu


In [9]:
# ── MODEL 1: LSTM ──────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# ── MODEL 2: GRU ───────────────────────────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])


# ── MODEL 3: TCN ───────────────────────────────────────────────────────────────
class CausalConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation)
        self.pad  = pad
        self.relu = nn.ReLU()
        self.norm = nn.BatchNorm1d(out_ch)
        self.res  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        out = self.conv(x)[:, :, :-self.pad] if self.pad > 0 else self.conv(x)
        return self.relu(self.norm(out) + self.res(x))

class TCNModel(nn.Module):
    def __init__(self, input_size, num_channels=64, kernel_size=3, num_levels=4):
        super().__init__()
        layers, in_ch = [], input_size
        for i in range(num_levels):
            layers.append(CausalConvBlock(in_ch, num_channels, kernel_size, 2 ** i))
            in_ch = num_channels
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels, 1)
    def forward(self, x):
        return self.fc(self.network(x.permute(0, 2, 1))[:, :, -1])

In [10]:
def get_batches(X, y, batch_size):
    for i in range(0, len(X), batch_size):
        yield X[i : i + batch_size], y[i : i + batch_size]


def train_and_evaluate(model, name):
    print(f"\n{'─'*55}\n  Training: {name}\n{'─'*55}")
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    loss_fn   = nn.L1Loss()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for bx, by in get_batches(X_train_t, y_train_t, BATCH_SIZE):
            optimizer.zero_grad()
            loss = loss_fn(model(bx), by)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            preds = scaler_y.inverse_transform(model(X_test_t).cpu().numpy())
            true  = scaler_y.inverse_transform(y_test_t.cpu().numpy())
            mae   = mean_absolute_error(true, preds)
            rmse  = math.sqrt(mean_squared_error(true, preds))

        mean_train = np.mean(train_losses)
        scheduler.step(mean_train)
        print(f"  Epoch {epoch}/{EPOCHS} | Train MAE: {mean_train:.4f} | Test MAE: {mae:.4f} | RMSE: {rmse:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    acc = np.mean(np.abs(true - preds) <= TOLERANCE) * 100
    return {"name": name, "model": model, "MAE": mae, "RMSE": rmse, "Accuracy": acc, "preds": preds, "true": true}

In [11]:
input_size = X_train_t.shape[2]
results = []
results.append(train_and_evaluate(LSTMModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE), "LSTM"))
results.append(train_and_evaluate(GRUModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE),  "GRU"))
results.append(train_and_evaluate(TCNModel(input_size).to(DEVICE),                           "TCN"))


───────────────────────────────────────────────────────
  Training: LSTM
───────────────────────────────────────────────────────
  Epoch 1/10 | Train MAE: 0.0833 | Test MAE: 0.7697 | RMSE: 9.8165 | LR: 0.001000
  Epoch 2/10 | Train MAE: 0.0733 | Test MAE: 0.6834 | RMSE: 8.9034 | LR: 0.001000
  Epoch 3/10 | Train MAE: 0.0674 | Test MAE: 0.6491 | RMSE: 8.4298 | LR: 0.001000
  Epoch 4/10 | Train MAE: 0.0643 | Test MAE: 0.6252 | RMSE: 8.1400 | LR: 0.001000
  Epoch 5/10 | Train MAE: 0.0621 | Test MAE: 0.6127 | RMSE: 7.9110 | LR: 0.001000
  Epoch 6/10 | Train MAE: 0.0605 | Test MAE: 0.6022 | RMSE: 7.7156 | LR: 0.001000
  Epoch 7/10 | Train MAE: 0.0592 | Test MAE: 0.5907 | RMSE: 7.5493 | LR: 0.001000
  Epoch 8/10 | Train MAE: 0.0581 | Test MAE: 0.5791 | RMSE: 7.3861 | LR: 0.001000
  Epoch 9/10 | Train MAE: 0.0571 | Test MAE: 0.5700 | RMSE: 7.2411 | LR: 0.001000
  Epoch 10/10 | Train MAE: 0.0562 | Test MAE: 0.5648 | RMSE: 7.1179 | LR: 0.001000

────────────────────────────────────────────────

In [12]:
print("\n" + "═"*62)
print(f"  MODEL COMPARISON  —  Target: {TARGET_KPI}")
print("═"*62)
print(f"  {'Model':<10} {'MAE':>10} {'RMSE':>10} {'Acc (±{:.1f})'.format(TOLERANCE):>14}")
print("  " + "─"*56)
best = min(results, key=lambda r: r["MAE"])
for r in results:
    marker = " ← BEST" if r["name"] == best["name"] else ""
    print(f"  {r['name']:<10} {r['MAE']:>10.4f} {r['RMSE']:>10.4f} {r['Accuracy']:>13.2f}%{marker}")
print("═"*62)


══════════════════════════════════════════════════════════════
  MODEL COMPARISON  —  Target: one_way_latency_ms
══════════════════════════════════════════════════════════════
  Model             MAE       RMSE     Acc (±2.0)
  ────────────────────────────────────────────────────────
  LSTM           0.5648     7.1179         98.61% ← BEST
  GRU            0.6078     7.9163         98.61%
  TCN            0.5895     6.6629         98.01%
══════════════════════════════════════════════════════════════


In [13]:
# === PREDICTION TEST ===
# Test the best model on 4 sample rows from the test set
# Each row = a sequence of SEQ_LEN=12 steps -> predicts the next value of one_way_latency_ms

import pandas as pd
import torch
import numpy as np

# Pick 4 evenly-spaced indices from the test set
sample_indices = [0, len(X_test)//3, 2*len(X_test)//3, len(X_test)-1]

best_model = best["model"]
best_model.eval()

rows = []
with torch.no_grad():
    for idx in sample_indices:
        # Raw sequence (before scaling) — shape (SEQ_LEN, n_features)
        seq_raw = X_test[idx]                                           # (12, 7)

        # Scale and convert to tensor
        seq_scaled = scaler_X.transform(seq_raw).astype(np.float32)
        seq_t      = torch.tensor(seq_scaled[np.newaxis, :, :], device=DEVICE)  # (1, 12, 7)

        # Predict
        pred_scaled = best_model(seq_t).cpu().numpy()                   # (1, 1)
        pred_val    = scaler_y.inverse_transform(pred_scaled)[0, 0]

        # Actual next-step value
        true_val = float(y_test[idx, 0])

        error    = abs(pred_val - true_val)
        within   = "✓" if error <= TOLERANCE else "✗"

        rows.append({
            "Sample index":        idx,
            f"Last known {TARGET_KPI}": round(seq_raw[-1, FEATURE_COLS.index(TARGET_KPI)], 4),
            "Predicted (t+1)":     round(pred_val, 4),
            "Actual (t+1)":        round(true_val, 4),
            "Abs Error":           round(error, 4),
            f"Within ±{TOLERANCE}": within,
        })

df_test = pd.DataFrame(rows)

print(f"Best model  : {best['name']}")
print(f"Target KPI  : {TARGET_KPI}")
print(f"Tolerance   : ±{TOLERANCE} ms")
print()
display(df_test)


Best model  : LSTM
Target KPI  : one_way_latency_ms
Tolerance   : ±2.0 ms



,Sample index,Last known one_way_latency_ms,Predicted (t+1),Actual (t+1),Abs Error,Within ±2.0
0,0,2.474500,2.4388,2.5084,0.0696,✓
1,14034,3.334400,3.1362,3.0841,0.0521,✓
2,28068,46.618198,7.9008,8.8396,0.9388,✓
3,42102,2.793000,2.7144,2.8602,0.1458,✓


In [17]:
# =====================================================================
# Export best model + scalers to PKL_models/
# =====================================================================
import os, joblib

pkl_dir = os.path.normpath(os.path.join(os.getcwd(), "..", "kafka_streaming", "PKL_models"))
os.makedirs(pkl_dir, exist_ok=True)

best_model = best["model"]
best_model.eval()

# Save PyTorch model (full object for easy reload)
model_path = os.path.join(pkl_dir, f"{TARGET_KPI}_best_model.pt")
torch.save(best_model, model_path)
print(f"Model  saved : {model_path}")

# Save scalers
scaler_X_path = os.path.join(pkl_dir, f"{TARGET_KPI}_scaler_X.pkl")
scaler_y_path = os.path.join(pkl_dir, f"{TARGET_KPI}_scaler_y.pkl")
joblib.dump(scaler_X, scaler_X_path)
joblib.dump(scaler_y, scaler_y_path)
print(f"scaler_X saved : {scaler_X_path}")
print(f"scaler_y saved : {scaler_y_path}")

# Save feature column list
feat_path = os.path.join(pkl_dir, f"{TARGET_KPI}_feature_cols.pkl")
joblib.dump(FEATURE_COLS, feat_path)
print(f"Feature cols saved : {feat_path}")



Model  saved : /Users/ayoubkallel/PFA2/-Intelligent-Anomaly-Monitoring-in-5G-networks/kafka_streaming/PKL_models/one_way_latency_ms_best_model.pt
scaler_X saved : /Users/ayoubkallel/PFA2/-Intelligent-Anomaly-Monitoring-in-5G-networks/kafka_streaming/PKL_models/one_way_latency_ms_scaler_X.pkl
scaler_y saved : /Users/ayoubkallel/PFA2/-Intelligent-Anomaly-Monitoring-in-5G-networks/kafka_streaming/PKL_models/one_way_latency_ms_scaler_y.pkl
Feature cols saved : /Users/ayoubkallel/PFA2/-Intelligent-Anomaly-Monitoring-in-5G-networks/kafka_streaming/PKL_models/one_way_latency_ms_feature_cols.pkl


In [35]:
# === EXPORT ALL PREDICTIONS FOR POWER BI ===
best_model = best["model"]
best_model.eval()

predictions = []

with torch.no_grad():
    for idx in range(len(X_test)):
        seq_raw = X_test[idx]
        seq_scaled = scaler_X.transform(seq_raw).astype(np.float32)
        seq_t = torch.tensor(seq_scaled[np.newaxis, :, :], device=DEVICE)

        pred_scaled = best_model(seq_t).cpu().numpy()
        pred_val = float(scaler_y.inverse_transform(pred_scaled)[0, 0])
        true_val = float(y_test[idx, 0])

        mae = best["MAE"]
        row_idx = train_n + SEQ_LEN + idx

        predictions.append({
            "timestamp": str(df.iloc[row_idx]["timestamp"]),
            "slice_type": df.iloc[row_idx]["slice_type"],
            "kpi_name": TARGET_KPI,
            "actual_value": round(true_val, 4),
            "predicted_value": round(pred_val, 4),
            "confidence_upper": round(pred_val + 1.5 * mae, 4),
            "confidence_lower": round(max(0, pred_val - 1.5 * mae), 4),
            "model_name": best["name"]
        })

df_pred = pd.DataFrame(predictions)
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_pred.to_csv(f"{OUTPUT_DIR}/{TARGET_KPI}_predictions.csv", index=False)
print(f"Exported {len(df_pred)} predictions to {OUTPUT_DIR}/{TARGET_KPI}_predictions.csv")
df_pred.head()

Exported 42103 predictions to ../outputs/latency/one_way_latency_ms_predictions.csv


,timestamp,slice_type,kpi_name,actual_value,predicted_value,confidence_upper,confidence_lower,model_name
0,2025-08-07 19:20:00,URLLC,one_way_latency_ms,2.5084,2.4388,3.2860,1.5917,LSTM
1,2025-08-07 19:25:00,URLLC,one_way_latency_ms,2.5321,2.4674,3.3146,1.6203,LSTM
2,2025-08-07 19:30:00,URLLC,one_way_latency_ms,2.4498,2.4899,3.3371,1.6428,LSTM
3,2025-08-07 19:35:00,URLLC,one_way_latency_ms,2.4323,2.4898,3.3369,1.6426,LSTM
4,2025-08-07 19:40:00,URLLC,one_way_latency_ms,2.5015,2.4677,3.3149,1.6206,LSTM
